# SAM3 — 노란 바 이물질·파손 탐지

세 가지 신호를 조합해서 이상을 판정한다.

| 신호 | 정상 | 이상 |
|---|---|---|
| **이물질 겹침** | 바 마스크와 이물질 마스크 교집합 없음 | IoU > 임계값 |
| **Solidity** | 마스크 면적 / 볼록껍질 면적 ≈ 1.0 | < 0.85 |
| **연결 성분** | 바 1개 = 덩어리 1개 | 2개 이상으로 분리 |

In [1]:
import cv2
import numpy as np
from pathlib import Path
from datetime import datetime
from ultralytics.models.sam import SAM3SemanticPredictor

In [ ]:
VIDEO_PATH   = "videos/test.mp4"
_ts          = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_VIDEO = f"sam3_foreign_{_ts}.mp4"

# ── 프롬프트 그룹 ──────────────────────────────────────────────
BAR_PROMPTS     = ["yellow bar", "yellow safety bar"]
FOREIGN_PROMPTS = ["foreign object", "obstacle", "debris"]
ALL_PROMPTS     = BAR_PROMPTS + FOREIGN_PROMPTS

BAR_IDS     = list(range(len(BAR_PROMPTS)))
FOREIGN_IDS = list(range(len(BAR_PROMPTS), len(ALL_PROMPTS)))

# ── ROI: 하단 몇 % 만 분석 ────────────────────────────────────
ROI_Y_RATIO = 0.6   # 0.6 → 프레임 60% 지점부터 하단 40%만 사용

# ── 이상 판정 임계값 ───────────────────────────────────────────
OVERLAP_THRESH  = 0.05
SOLIDITY_THRESH = 0.85
MIN_COMPONENTS  = 2

TARGET_FPS = 5
CONF       = 0.25

print(f"바 프롬프트    : {BAR_PROMPTS}")
print(f"이물질 프롬프트: {FOREIGN_PROMPTS}")
print(f"ROI            : 하단 {(1-ROI_Y_RATIO)*100:.0f}%  (y >= {ROI_Y_RATIO*100:.0f}%)")
print(f"출력 파일      : {OUTPUT_VIDEO}")

## Step 1 — SAM3 초기화

In [3]:
overrides = dict(
    conf=CONF,
    task="segment",
    mode="predict",
    model="sam3.pt",
    half=True,
    save=False,
)
predictor = SAM3SemanticPredictor(overrides=overrides)
print("초기화 완료")

초기화 완료


## Step 2 — 프레임 추출

In [4]:
cap = cv2.VideoCapture(VIDEO_PATH)
src_fps = 60.0
fh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fw = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
step = max(1, round(src_fps / TARGET_FPS))

frames = []
idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if idx % step == 0:
        frames.append((idx, frame))
    idx += 1
cap.release()

print(f"{len(frames)} 프레임 추출  ({fw}×{fh})  step={step}")

55 프레임 추출  (1080×1920)  step=12


## Step 3 — 판별 함수 정의

In [ ]:
def mask_overlap_ratio(mask_a, mask_b):
    """두 마스크의 IoU (교집합 / 합집합)"""
    inter = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    if union == 0:
        return 0.0
    return float(inter) / float(union)


def analyze_bar(mask_u8, foreign_masks):
    """이물질 겹침만 판별 (shape/split 제거)"""
    max_overlap = max(
        (mask_overlap_ratio(mask_u8, fm) for fm in foreign_masks),
        default=0.0,
    )

    reasons = []
    if max_overlap >= OVERLAP_THRESH:
        reasons.append(f"FOREIGN({max_overlap:.2f})")

    return dict(overlap=max_overlap, reasons=reasons)

print("판별 함수 로드 완료")

## Step 4 — 추론 & 이상 탐지 영상 저장

In [ ]:
writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*"mp4v"),
    TARGET_FPS,
    (fw, fh),
)

COLOR_BAR     = (  0, 220, 255)
COLOR_ANOMALY = (  0,   0, 255)
COLOR_FOREIGN = (100,   0, 255)

anomaly_log = []

for proc_idx, (orig_idx, frame) in enumerate(frames):
    roi_y = int(fh * ROI_Y_RATIO)

    # ── 하단 ROI만 잘라서 SAM3에 넘김 ──────────────────────────
    roi = frame[roi_y:, :]
    rh, rw = roi.shape[:2]   # ROI 해상도

    # ── 프롬프트 그룹 ──────────────────────────────────────────────
    BAR_PROMPTS     = ["yellow bar", "yellow safety bar"]
    FOREIGN_PROMPTS = ["foreign object", "obstacle", "debris"]
    ALL_PROMPTS     = BAR_PROMPTS + FOREIGN_PROMPTS
    
    predictor.set_image(roi)
    results = predictor(text=ALL_PROMPTS)

    out = frame.copy()
    # ROI 경계선 표시
    cv2.line(out, (0, roi_y), (fw - 1, roi_y), (0, 255, 255), 2)

    frame_reasons = []

    if results and results[0].masks is not None:
        r       = results[0]
        masks   = r.masks.data.cpu().numpy().astype(np.uint8)
        cls_ids = r.boxes.cls.cpu().numpy().astype(int) if r.boxes is not None else []

        # 마스크를 ROI 해상도로 리사이즈
        masks_roi = [cv2.resize(m, (rw, rh), interpolation=cv2.INTER_NEAREST)
                     for m in masks]

        bar_masks     = [m for m, c in zip(masks_roi, cls_ids) if c in BAR_IDS]
        foreign_masks = [m for m, c in zip(masks_roi, cls_ids) if c in FOREIGN_IDS]

        # 이물질 오버레이 (ROI 영역에만)
        for fm in foreign_masks:
            overlay = out.copy()
            overlay[roi_y:][fm > 0] = COLOR_FOREIGN
            cv2.addWeighted(overlay, 0.45, out, 0.55, 0, out)

        # 바 마스크 분석 & 오버레이
        for bm in bar_masks:
            result = analyze_bar(bm, foreign_masks)
            color  = COLOR_ANOMALY if result["reasons"] else COLOR_BAR

            overlay = out.copy()
            overlay[roi_y:][bm > 0] = color
            cv2.addWeighted(overlay, 0.45, out, 0.55, 0, out)

            if result["reasons"]:
                ys, xs = np.where(bm > 0)
                cx = int(xs.mean())
                cy = int(ys.mean()) + roi_y   # 원본 좌표로 복원
                label = " | ".join(result["reasons"])
                cv2.putText(out, label, (max(0, cx - 60), cy),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
                frame_reasons.extend(result["reasons"])

    # ── HUD ──────────────────────────────────────────────────────
    hud = out.copy()
    cv2.rectangle(hud, (0, 0), (fw, 90), (0, 0, 0), -1)
    cv2.addWeighted(hud, 0.50, out, 0.50, 0, out)

    status       = "ANOMALY" if frame_reasons else "OK"
    status_color = (0, 0, 255) if frame_reasons else (0, 200, 0)
    cv2.putText(out, f"[{status}]  frame={orig_idx}  ({proc_idx+1}/{len(frames)})",
                (12, 35), cv2.FONT_HERSHEY_SIMPLEX, 1.0, status_color, 2, cv2.LINE_AA)
    cv2.putText(out, f"bars={len(bar_masks)}  foreign={len(foreign_masks)}  ROI=bottom{(1-ROI_Y_RATIO)*100:.0f}%",
                (12, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (200, 200, 200), 1, cv2.LINE_AA)

    if status == "ANOMALY":
        cv2.rectangle(out, (0, 0), (fw - 1, fh - 1), (0, 0, 255), 6)
        anomaly_log.append((orig_idx, list(set(frame_reasons))))

    writer.write(out)

    if proc_idx % 10 == 0:
        print(f"  [{proc_idx+1:3d}/{len(frames)}] frame={orig_idx}  {status}  "
              f"bars={len(bar_masks)}  foreign={len(foreign_masks)}")

writer.release()
print(f"\n완료: {OUTPUT_VIDEO}")
print(f"이상 프레임: {len(anomaly_log)}건")

## Step 5 — 이상 로그 요약

In [7]:
if not anomaly_log:
    print("이상 없음")
else:
    print(f"{'프레임':>8}  원인")
    print("-" * 40)
    for fidx, reasons in anomaly_log:
        print(f"{fidx:>8}  {', '.join(reasons)}")
    print(f"\n이상률: {len(anomaly_log)/len(frames)*100:.1f}%")

     프레임  원인
----------------------------------------
       0  SPLIT(18), SPLIT(2), SHAPE(0.06), SHAPE(0.79)
      12  SHAPE(0.80), SHAPE(0.06), SPLIT(6), SHAPE(0.11), SHAPE(0.08), SPLIT(2), SPLIT(4), SHAPE(0.42)
      24  SHAPE(0.04), SHAPE(0.14), SHAPE(0.06), SPLIT(12), SHAPE(0.10), SPLIT(4), SPLIT(18), SHAPE(0.76), SPLIT(7)
      36  SPLIT(4), SHAPE(0.04), SHAPE(0.14), SPLIT(10)
      48  SPLIT(6), SPLIT(2), SHAPE(0.07), SPLIT(18), SHAPE(0.79)
      60  SHAPE(0.09), SHAPE(0.08), SPLIT(3), SPLIT(7)
      72  SPLIT(4), SHAPE(0.38)
      84  SPLIT(9), SHAPE(0.11)
      96  SHAPE(0.04), SHAPE(0.10), SHAPE(0.81), SHAPE(0.20), SPLIT(2), SHAPE(0.71), SHAPE(0.82), SPLIT(7), SPLIT(16)
     108  SPLIT(3), SPLIT(2), SHAPE(0.30), SHAPE(0.77)
     120  SHAPE(0.27), SHAPE(0.04), SHAPE(0.78), SHAPE(0.43), SPLIT(2), SPLIT(4), SPLIT(17)
     132  SHAPE(0.08), SPLIT(2), SHAPE(0.07), SPLIT(4), SHAPE(0.72), SPLIT(10)
     144  SHAPE(0.04), SPLIT(3), SHAPE(0.85), SPLIT(2), SHAPE(0.21), SHAPE(0.13), SPL